# Project Lab, Part 2: An Analytics Case Study and a Query-Optimization Lab

Notebook for **Chapter 16** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: A 20-Question SQL Analytics Case Study with a Findings Report

In [ ]:
import sqlite3, random, sys
from datetime import date, timedelta
random.seed(5)
con = sqlite3.connect(sys.argv[1])
con.executescript("""
DROP TABLE IF EXISTS appointments; DROP TABLE IF EXISTS patients; DROP TABLE IF EXISTS doctors;
CREATE TABLE doctors (doctor_id INTEGER PRIMARY KEY, name TEXT NOT NULL, department TEXT NOT NULL);
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, age INTEGER NOT NULL, gender TEXT NOT NULL, city TEXT NOT NULL);
CREATE TABLE appointments (
  appt_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients,
  doctor_id INTEGER NOT NULL REFERENCES doctors,
  booked_on TEXT NOT NULL, appt_date TEXT NOT NULL,
  status TEXT NOT NULL CHECK (status IN ('attended','no_show','cancelled')),
  wait_minutes INTEGER, fee REAL NOT NULL);
""")
depts = ["General","Cardiology","Orthopedics","Pediatrics"]
con.executemany("INSERT INTO doctors VALUES (?,?,?)", [(i, f"Dr. {chr(64+i)}", depts[(i-1)%4]) for i in range(1,13)])
con.executemany("INSERT INTO patients VALUES (?,?,?,?)",
  [(i, random.randint(1,85), random.choice("MF"), random.choice(["North","South","East","West"])) for i in range(1,1501)])
fees = {"General":300,"Cardiology":800,"Orthopedics":600,"Pediatrics":400}
rows = []
for a in range(1, 6001):
    d = random.randint(1,12); dep = depts[(d-1)%4]
    lead = random.choice([0,1,2,3,5,7,10,14,21,30])
    appt = date(2024,1,1) + timedelta(days=random.randint(0,364))
    booked = appt - timedelta(days=lead)
    p_no = 0.06 + 0.012*lead + (0.05 if dep == "Pediatrics" else 0)
    r = random.random()
    if r < p_no: st = "no_show"
    elif r < p_no + 0.08: st = "cancelled"
    else: st = "attended"
    wait = max(0, int(random.gauss(22 + 3*(d%4) + (10 if appt.weekday() == 0 else 0), 9))) if st == "attended" else None
    rows.append((a, random.randint(1,1500), d, booked.isoformat(), appt.isoformat(), st, wait, fees[dep]))
con.executemany("INSERT INTO appointments VALUES (?,?,?,?,?,?,?,?)", rows); con.commit()
print(con.execute("select count(*) from appointments").fetchone())

(6000,)


## 16.2 Project: A Query-Optimization Lab, from Slow to Fast

In [ ]:
import sqlite3, random, sys
from datetime import date, timedelta
random.seed(1)
con = sqlite3.connect(sys.argv[1])
con.executescript("""
DROP TABLE IF EXISTS orders; DROP TABLE IF EXISTS customers;
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, email TEXT);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);
""")
cities = ["Pune","Delhi","Mumbai","Chennai","Kolkata","Jaipur","Surat","Indore"]
con.executemany("INSERT INTO customers VALUES (?,?,?,?)",
  [(i, f"Customer {i}", random.choice(cities), f"user{i}@example.com") for i in range(1, 20001)])
base = date(2023,1,1); st = ["paid"]*85 + ["refunded"]*5 + ["cancelled"]*10
rows = [(i, random.randint(1,20000), (base+timedelta(days=random.randint(0,729))).isoformat(),
         random.choice(st), round(random.uniform(5,500),2)) for i in range(1, 200001)]
con.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", rows); con.commit()
print(con.execute("select count(*) from orders").fetchone(), con.execute("select count(*) from customers").fetchone())

(200000,) (20000,)


In [ ]:
import sqlite3, time
con = sqlite3.connect("lab.db")
def timeit(sql, n=5):
    best = 1e9
    for _ in range(n):
        t = time.perf_counter(); rows = con.execute(sql).fetchall()
        best = min(best, time.perf_counter() - t)
    return best * 1000, rows
def report(label, sql, n=5):
    ms, rows = timeit(sql, n)
    print(f"[{label}] best of {n}: {ms:9.2f} ms   result={rows[0] if rows else None}")
    for r in con.execute("EXPLAIN QUERY PLAN " + sql): print("    plan:", r[3])

Q1 = "SELECT * FROM orders WHERE customer_id = 777"
Q2_BEFORE = "SELECT COUNT(*) FROM orders WHERE substr(order_date,1,7)='2024-03'"
Q2_AFTER  = "SELECT COUNT(*) FROM orders WHERE order_date>='2024-03-01' AND order_date<'2024-04-01'"
Q3_BEFORE = ("SELECT COUNT(*) FROM customers c WHERE c.customer_id<=300 AND "
             "(SELECT COUNT(*) FROM orders o WHERE o.customer_id=c.customer_id AND o.status='paid')>0")
Q3_AFTER  = ("SELECT COUNT(*) FROM customers c WHERE c.customer_id<=300 AND "
             "EXISTS (SELECT 1 FROM orders o WHERE o.customer_id=c.customer_id AND o.status='paid')")
Q4 = ("SELECT order_id,order_date,amount FROM orders WHERE status='paid' "
      "ORDER BY order_date DESC, order_id DESC LIMIT 10")
Q5 = ("SELECT customer_id, SUM(amount) s FROM orders WHERE status='paid' AND order_date>='2024-01-01' "
      "GROUP BY customer_id ORDER BY s DESC LIMIT 5")

## 16.3 Worked Extensions: The Lead-Time Profile, a Risk Score and How Big a Test Must Be

In [ ]:
import math
# baseline no-show rate for bookings made 14+ days ahead: 33.8 percent (550 of 1628)
p1 = 550 / 1628
p2 = p1 - 0.05            # we hope reminders cut it by 5 points
z_alpha, z_beta = 1.96, 0.8416      # 95 percent confidence, 80 percent power
d = p1 - p2
n_arm = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / d ** 2
print("baseline %.1f%%, target %.1f%%" % (100 * p1, 100 * p2))
print("needed per arm: %d, total: %d" % (math.ceil(n_arm), 2 * math.ceil(n_arm)))
per_week = 1628 / 52
print("such bookings per week: %.0f, weeks to fill: %.0f" % (per_week, 2 * math.ceil(n_arm) / per_week))

baseline 33.8%, target 28.8%
needed per arm: 1346, total: 2692
such bookings per week: 31, weeks to fill: 86


## Chapter recap

## Practice set

Problem 16.P.1 (easy) 1,057 of 6,000 appointments are no-shows. What percent is that? **Answer:** 1057 / 6000 = 0.17617, so 17.6%.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch16.html